# Actividad 2: Despliegue y Procesamiento de Datos en Databricks CE
**Dataset seleccionado:** Video Game Sales (Kaggle)

## 1. Diseño del Esquema y Diccionario de Datos
El dataset contiene un registro histórico de las ventas de videojuegos a nivel mundial. A continuación, se describen las entidades, campos clave, tipos de datos y su nulabilidad.

| Campo | Tipo de Dato | Nulabilidad | Llave | Descripción |
| :--- | :--- | :--- | :--- | :--- |
| `Rank` | Integer | No Nulo | PK | Identificador único y ranking global de ventas. |
| `Name` | String | No Nulo | - | Nombre del videojuego. |
| `Platform` | String | No Nulo | - | Plataforma o consola de lanzamiento. |
| `Year` | Integer | Permite Nulos| - | Año de lanzamiento del juego. |
| `Genre` | String | No Nulo | - | Género del videojuego (Acción, Deportes, etc.). |
| `Publisher` | String | Permite Nulos| - | Empresa que publica el juego. |
| `Global_Sales`| Double | No Nulo | - | Ventas totales a nivel mundial (en millones de copias). |

## Diagrama Lógico del Esquema (Mermaid)

```mermaid
erDiagram
    VIDEOJUEGO {
        int Rank PK
        string Name
        string Platform
        int Year
        string Genre
        string Publisher
        double Global_Sales
    }

In [0]:
# Importar los tipos de datos necesarios de PySpark
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Definición del esquema (StructType) para el dataset de Videojuegos
video_games_schema = StructType([
    StructField("Rank", IntegerType(), nullable=False),
    StructField("Name", StringType(), nullable=False),
    StructField("Platform", StringType(), nullable=False),
    StructField("Year", IntegerType(), nullable=True),
    StructField("Genre", StringType(), nullable=False),
    StructField("Publisher", StringType(), nullable=True),
    StructField("Global_Sales", DoubleType(), nullable=False)
])

print("Esquema definido exitosamente en PySpark.")

Esquema definido exitosamente en PySpark.


## 2. Configuración y Evidencia de la Infraestructura en Databricks CE
Para el procesamiento de los datos, se utiliza la infraestructura de la versión comunitaria de Databricks. Debido a las características de este entorno, la configuración es la siguiente:

* **Tipo de Clúster:** Single Node (Un solo nodo, el driver actúa también como worker).
* **Autoscaling:** Deshabilitado (limitación de la versión gratuita CE).
* **Estructura de Almacenamiento:** DBFS (Databricks File System).

In [0]:
import sys

# 1. Evidencia de las versiones de Python y Spark
print(f"Versión de Python: {sys.version.split(' ')[0]}")
print(f"Versión actual de Spark: {spark.version}")

print("\n--- Configuraciones del Clúster ---")
try:
    # Intento tradicional exigido en la rúbrica
    configuraciones = spark.sparkContext.getConf().getAll()
    for conf in configuraciones:
        if "spark.databricks" in conf[0] or "spark.master" in conf[0]:
            print(f"- {conf[0]}: {conf[1]}")
except Exception as e:
    # Alternativa segura para entornos modernos Serverless/Unity Catalog
    print("Nota: Acceso directo a sparkContext restringido por el entorno Serverless.")
    print("Extrayendo configuraciones mediante la interfaz SQL:")
    
    # Extraemos y filtramos la configuración con SQL
    df_conf = spark.sql("SET")
    df_conf.filter(df_conf.key.rlike("spark.master|spark.databricks")).show(truncate=False)

Versión de Python: 3.12.3
Versión actual de Spark: 4.1.0

--- Configuraciones del Clúster ---
Nota: Acceso directo a sparkContext restringido por el entorno Serverless.
Extrayendo configuraciones mediante la interfaz SQL:
+----------------------------------+-----+
|key                               |value|
+----------------------------------+-----+
|spark.databricks.execution.timeout|9000 |
+----------------------------------+-----+



## 3. Obtención de Datos y Creación de Tabla
Para este paso, se utilizó la **Opción B**: descarga manual del dataset `vgsales.csv` desde Kaggle y carga al sistema de almacenamiento (Volumes).

A continuación, se lee el archivo CSV cargándolo en un DataFrame de PySpark (aplicando el esquema estricto diseñado en el paso 1). Finalmente, se persiste la información creando una tabla para poder consultarla con SQL.

In [0]:
# 1. Definir la ruta donde subimos nuestro archivo en el Volume
ruta_archivo = "/Volumes/workspace/default/datos_actividad/vgsales.csv"

# 2. Leer el CSV aplicando el esquema (video_games_schema)
df_juegos = spark.read.format("csv") \
    .option("header", "true") \
    .schema(video_games_schema) \
    .load(ruta_archivo)

# 3. Mostrar las primeras 5 filas para confirmar la lectura
print("Muestra de los datos ingeridos desde el Volume:")
df_juegos.show(5)

# 4. Persistir los datos: Crear una tabla administrada
nombre_tabla = "video_game_sales"
df_juegos.write.mode("overwrite").saveAsTable(nombre_tabla)

print(f"\nTabla '{nombre_tabla}' guardada exitosamente.")

Muestra de los datos ingeridos desde el Volume:
+----+--------------------+--------+----+------------+---------+------------+
|Rank|                Name|Platform|Year|       Genre|Publisher|Global_Sales|
+----+--------------------+--------+----+------------+---------+------------+
|   1|          Wii Sports|     Wii|2006|      Sports| Nintendo|       41.49|
|   2|   Super Mario Bros.|     NES|1985|    Platform| Nintendo|       29.08|
|   3|      Mario Kart Wii|     Wii|2008|      Racing| Nintendo|       15.85|
|   4|   Wii Sports Resort|     Wii|2009|      Sports| Nintendo|       15.75|
|   5|Pokemon Red/Pokem...|      GB|1996|Role-Playing| Nintendo|       11.27|
+----+--------------------+--------+----+------------+---------+------------+
only showing top 5 rows

Tabla 'video_game_sales' guardada exitosamente.


In [0]:
%sql
-- Validación de la creación de la tabla y sus metadatos
DESCRIBE TABLE video_game_sales;

col_name,data_type,comment
Rank,int,null
Name,string,null
Platform,string,null
Year,int,null
Genre,string,null
Publisher,string,null
Global_Sales,double,null


## 4. Validaciones en Spark y SQL
En esta sección se comparan las formas de interactuar con los datos utilizando el paradigma programático (PySpark DataFrames) frente al paradigma declarativo (Spark SQL). 

**Validación de Metadatos y Estadísticas Descriptivas**
Se comprueba que el esquema y las estadísticas básicas (conteos, promedios) sean coherentes en ambas APIs.

In [0]:
print("--- 1. Metadatos del Esquema en PySpark ---")
df_juegos.printSchema()

print("\n--- 2. Descripción Estadística de los Datos (PySpark) ---")
# describe() calcula el conteo, promedio, desviación estándar, min y max
df_juegos.describe().show()

--- 1. Metadatos del Esquema en PySpark ---
root
 |-- Rank: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Platform: string (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)
 |-- Publisher: string (nullable = true)
 |-- Global_Sales: double (nullable = true)


--- 2. Descripción Estadística de los Datos (PySpark) ---
+-------+-----------------+--------------------+--------+------------------+--------+---------------+------------------+
|summary|             Rank|                Name|Platform|              Year|   Genre|      Publisher|      Global_Sales|
+-------+-----------------+--------------------+--------+------------------+--------+---------------+------------------+
|  count|            16598|               16598|   16598|             16327|   16598|          16598|             16598|
|   mean|8300.605253645017|              1942.0|  2600.0|2006.4064433147546|    NULL|           NULL|0.2646674298108279|
| stddev|4791.

In [0]:
%sql
-- Validación de metadatos usando Spark SQL
DESCRIBE TABLE video_game_sales;

col_name,data_type,comment
Rank,int,null
Name,string,null
Platform,string,null
Year,int,null
Genre,string,null
Publisher,string,null
Global_Sales,double,null


**Consultas Analíticas (SELECT, GROUP BY y LIMIT)**
Para validar el procesamiento, se plantea la siguiente consulta de negocio: *¿Cuáles son las 5 plataformas con mayores ventas globales acumuladas?* Se resolverá primero con la API de PySpark y luego con una consulta de Spark SQL para demostrar la equivalencia absoluta de los resultados.

In [0]:
from pyspark.sql.functions import sum, desc

print("--- Top 5 Plataformas con más Ventas (PySpark) ---")

# Consulta usando métodos de DataFrame
df_juegos.groupBy("Platform") \
    .agg(sum("Global_Sales").alias("Total_Ventas_Millones")) \
    .orderBy(desc("Total_Ventas_Millones")) \
    .limit(5) \
    .show()

--- Top 5 Plataformas con más Ventas (PySpark) ---
+--------+---------------------+
|Platform|Total_Ventas_Millones|
+--------+---------------------+
|    X360|    601.0500000000005|
|     PS2|    583.8399999999945|
|     Wii|   507.70999999999964|
|     PS3|    392.2600000000002|
|      DS|    390.7099999999996|
+--------+---------------------+



In [0]:
%sql
-- La misma consulta exacta pero usando Spark SQL
SELECT Platform, SUM(Global_Sales) AS Total_Ventas_Millones
FROM video_game_sales
GROUP BY Platform
ORDER BY Total_Ventas_Millones DESC
LIMIT 5;

Platform,Total_Ventas_Millones
X360,601.0499999999992
PS2,583.8399999999925
Wii,507.7099999999991
PS3,392.2599999999998
DS,390.7099999999977


## 5. Ventajas y Desventajas: SQL vs Spark (PySpark)
A continuación, se presenta el análisis comparativo analizando el uso de ambos enfoques durante el desarrollo de esta actividad:

| Criterio | Spark SQL (Declarativo) | PySpark DataFrames (Programático) |
| :--- | :--- | :--- |
| **Facilidad de uso** | **Alta:** Utiliza una sintaxis estándar y familiar para analistas de BI y bases de datos relacionales. | **Media:** Requiere conocimientos de programación en Python y familiaridad con la API de Spark. |
| **Construcción de Pipelines** | **Compleja:** Concatenar múltiples transformaciones en consultas SQL largas puede volverse difícil de leer y mantener. | **Excelente:** Permite encadenar métodos de forma limpia (`.groupBy().agg().orderBy()`), facilitando la modularidad. |
| **Flexibilidad y Control** | **Limitada:** Crear funciones personalizadas complejas (UDFs) o lógicas condicionales avanzadas es más rígido. | **Alta:** Ofrece control total sobre el flujo de datos, integración con APIs ricas y manipulación avanzada de RDDs. |
| **Ecosistema e Integración** | Se integra de forma directa con herramientas de visualización de Business Intelligence (PowerBI, Tableau). | Es la puerta de entrada para analítica avanzada, procesamiento en tiempo real (Streaming) y Machine Learning (MLlib). |
| **Rendimiento** | Ambos enfoques comparten el mismo motor de optimización subyacente (**Catalyst Optimizer**), por lo que ejecutan los planes de cómputo distribuidos con la misma eficiencia. |